In [1]:
import os, json, math, random, gc, time, ast, copy
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights, resnet18

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

print("✅ Imports successful")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

✅ Imports successful
✅ CUDA available: True


# BirdCLEF 2026 Training v7 - ResNet18 + EfficientNet-B0 (Fixed Ensemble)
## Fixes from v6 investigation:
- **Replaced weak PerchAudio** with proven **ResNet18** architecture (128-ch CNN was underfitting)
- **Removed SpecAugment** (time/freq masking proven to hurt: 0.648\u21920.559 in Phase 1)
- **Fixed soundscape CSV filename** (`train_soundscapes_labels.csv`, not `train_soundscape_labels.csv`)
- **Targeted soundscape extraction** for missing/underrepresented species only
- **Fixed EfficientNet model save bug** (shallow `.copy()` \u2192 `copy.deepcopy()`)
- **Added AUC metric tracking** per fold (not just loss)
- **Cosine annealing LR scheduler** for better convergence
- **Uniform 0.5 thresholds** (per-species F1 optimization proven not to generalize)

In [2]:
# === DATA PATHS & SPECIES ===
TRAIN_META_CSV = "/kaggle/input/competitions/birdclef-2026/train.csv"
TRAIN_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/train_audio"
TAXONOMY_CSV = "/kaggle/input/competitions/birdclef-2026/taxonomy.csv"

# Load complete species list from taxonomy.csv (234 species)
taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species = taxonomy_df["primary_label"].astype(str).tolist()

species_set = set(species)

print(f"\u2705 Loaded {len(species)} species from taxonomy.csv")
print(f"First 10 species: {species[:10]}")
print(f"Last 10 species: {species[-10:]}")

# Create species index mapping
idx = {lab: i for i, lab in enumerate(species)}
n_classes = len(species)

# Load training data
df = pd.read_csv(TRAIN_META_CSV)
print(f"\n\u2705 Loaded training data: {len(df)} samples")
print(f"Unique species in training: {df['primary_label'].nunique()}")

# Save species.json for inference
with open("/kaggle/working/species.json", "w") as f:
    json.dump(species, f)

print(f"\u2705 Saved species.json with {n_classes} species")

✅ Loaded 234 species from taxonomy.csv
First 10 species: ['1161364', '116570', '1176823', '1491113', '1595929', '209233', '22930', '22956', '22961', '22967']
Last 10 species: ['whnjay1', 'whtdov', 'whwpic1', 'y00678', 'yebcar', 'yebela1', 'yecmac', 'yecpar', 'yehcar1', 'yeofly1']

✅ Loaded training data: 35549 samples
Unique species in training: 206
✅ Saved species.json with 234 species


In [3]:
# === CONFIG ===
CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
    epochs=20,        # Increased from 15 for better convergence with cosine LR
    lr=1e-3,
    patience=7,       # Increased patience to allow cosine LR to help
    num_workers=0,    # Set to 0 to avoid DataLoader issues on Kaggle
    seed=42,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

print(f"Config: {CFG}")

Config: {'sr': 16000, 'n_mels': 64, 'n_fft': 1024, 'hop': 320, 'fmin': 60, 'seconds': 5, 'batch_size': 32, 'epochs': 20, 'lr': 0.001, 'patience': 7, 'num_workers': 0, 'seed': 42, 'device': 'cuda'}


In [4]:
# === HELPER FUNCTIONS ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ("", "[]"): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(len(species), dtype="float32")
    p = str(primary_id)
    if p in idx: y[idx[p]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in species_set:
            y[idx[sid]] = 1.0
    return y

def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr // 2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("\u2705 Helper functions defined")

✅ Helper functions defined


In [5]:
# === PRECOMPUTE MELS FROM TRAIN_AUDIO ===
print("Precomputing mels from train_audio\u2026")

MEL_OUT_DIR = "/kaggle/working/mels_v7"
os.makedirs(MEL_OUT_DIR, exist_ok=True)

for idx_, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
    mel_name = row["filename"].replace("/", "_") + ".npy"
    mel_path = Path(MEL_OUT_DIR) / mel_name

    # Skip if already computed
    if mel_path.exists():
        continue

    try:
        y, sr0 = sf.read(audio_path, always_2d=False)
    except Exception:
        print(f"Failed: {audio_path}")
        continue

    if sr0 != CFG["sr"]:
        y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])

    y = fixed_length_mono(y, CFG["sr"], CFG["seconds"])
    mel = logmel_from_wave(y, CFG["sr"])
    np.save(mel_path, mel)

print(f"\u2705 Mels saved from train_audio: {MEL_OUT_DIR}")

Precomputing mels from train_audio…


100%|██████████| 35549/35549 [46:47<00:00, 12.66it/s]

✅ Mels saved from train_audio: /kaggle/working/mels_v7


In [6]:
# === EXTRACT SEGMENTS FROM TRAIN_SOUNDSCAPES (Missing/Underrepresented Species) ===
# FIX v6 bug: correct CSV filename is train_soundscapes_labels.csv (with 's')
# FIX v6 bug: target missing/underrepresented species, not random segments

print("\n" + "=" * 70)
print("SOUNDSCAPE EXTRACTION: Targeting missing/underrepresented species")
print("=" * 70)

SOUNDSCAPE_DIR = "/kaggle/input/competitions/birdclef-2026/train_soundscapes"
# FIXED: correct filename (v6 used train_soundscape_labels.csv - missing the 's')
SOUNDSCAPE_ANNO_CSV = "/kaggle/input/competitions/birdclef-2026/train_soundscapes_labels.csv"

pseudo_df = pd.DataFrame()

try:
    soundscape_labels = pd.read_csv(SOUNDSCAPE_ANNO_CSV)
    print(f"Loaded {len(soundscape_labels)} soundscape annotations")
    print(f"Columns: {soundscape_labels.columns.tolist()}")

    # Identify species in train_audio
    train_audio_species = set(df["primary_label"].astype(str).unique())
    # Identify species in soundscapes
    soundscape_species_col = None
    for col in ["primary_label", "species", "bird_id"]:
        if col in soundscape_labels.columns:
            soundscape_species_col = col
            break
    if soundscape_species_col is None:
        soundscape_species_col = soundscape_labels.columns[1]
    print(f"Using soundscape species column: '{soundscape_species_col}'")

    soundscape_all_species = set(soundscape_labels[soundscape_species_col].astype(str).unique())

    # Species to prioritize: present in soundscapes but missing/rare in train_audio
    species_counts_audio = df["primary_label"].value_counts().to_dict()
    # Underrepresented = missing OR fewer than 5 clips in train_audio
    target_species = {
        sp for sp in soundscape_all_species
        if species_counts_audio.get(sp, 0) < 5
    }
    print(f"Species in soundscapes: {len(soundscape_all_species)}")
    print(f"Target species (missing/underrepresented in train_audio): {len(target_species)}")

    pseudo_data = []
    for idx_, row in tqdm(soundscape_labels.iterrows(), total=len(soundscape_labels),
                          desc="Processing soundscapes"):
        primary = str(row.get(soundscape_species_col, "unknown"))

        # Only process target species (missing/underrepresented)
        if primary not in target_species:
            continue

        # Determine audio file path
        soundscape_id = None
        for id_col in ["soundscape_id", "filename", "file_id"]:
            if id_col in row.index:
                soundscape_id = str(row[id_col])
                break
        if soundscape_id is None:
            soundscape_id = str(row.iloc[0])

        # Try common audio extensions
        audio_path = None
        for ext in [".ogg", ".wav", ".flac"]:
            candidate = Path(SOUNDSCAPE_DIR) / f"{soundscape_id}{ext}"
            if candidate.exists():
                audio_path = candidate
                break
        if audio_path is None:
            continue

        try:
            y, sr0 = sf.read(str(audio_path), always_2d=False)
        except Exception:
            continue

        if sr0 != CFG["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=CFG["sr"])
        if y.ndim == 2:
            y = y.mean(axis=1)
        y = y.astype(np.float32)

        segment_samples = CFG["sr"] * CFG["seconds"]
        total_segments = max(1, len(y) // segment_samples)

        # Sample up to 5 segments per soundscape for target species
        n_sample = min(5, total_segments)
        seg_indices = np.random.choice(total_segments, n_sample, replace=False)

        for seg_idx in seg_indices:
            start = seg_idx * segment_samples
            end = start + segment_samples
            if end > len(y):
                continue

            segment = y[start:end].copy()
            mel = logmel_from_wave(segment, CFG["sr"])

            mel_name = f"soundscape_{soundscape_id}_seg_{seg_idx}.npy"
            np.save(Path(MEL_OUT_DIR) / mel_name, mel)

            secondary = row.get("secondary_labels", "[]")
            pseudo_data.append({
                "filename": mel_name.replace(".npy", ""),
                "primary_label": primary,
                "secondary_labels": secondary,
            })

    if pseudo_data:
        pseudo_df = pd.DataFrame(pseudo_data)
        print(f"\u2705 Extracted {len(pseudo_df)} segments for {len(target_species)} target species")
    else:
        print("\u26a0\ufe0f  No soundscape segments extracted — proceeding with train_audio only")

except FileNotFoundError as e:
    print(f"\u26a0\ufe0f  Soundscape CSV not found: {e}")
    print("Continuing with train_audio only...")
except Exception as e:
    print(f"\u26a0\ufe0f  Soundscape processing failed: {e}")
    print("Continuing with train_audio only...")


SOUNDSCAPE EXTRACTION: Targeting missing/underrepresented species
Loaded 1478 soundscape annotations
Columns: ['filename', 'start', 'end', 'primary_label']
Using soundscape species column: 'primary_label'
Species in soundscapes: 251
Target species (missing/underrepresented in train_audio): 240


Processing soundscapes: 100%|██████████| 1478/1478 [00:00<00:00, 4796.80it/s]

⚠️  No soundscape segments extracted — proceeding with train_audio only


In [7]:
# === RESNET18 ARCHITECTURE (Replaces weak PerchAudio from v6) ===
# v6 used a 3-layer CNN (128 hidden) that underfits; ResNet18 is the proven 0.648 architecture.

class ResNet18Audio(nn.Module):
    """ResNet18-based classifier for mel spectrograms. Proven architecture (0.648 baseline)."""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        backbone = resnet18(weights=None)
        # Replace first conv: 3-channel RGB -> 1-channel mel spectrogram
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

print("\u2705 ResNet18Audio architecture defined")

✅ ResNet18Audio architecture defined


In [8]:
# === EFFICIENTNET-B0 ARCHITECTURE ===
class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        # Use pretrained ImageNet weights for better feature extraction
        self.model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        # Replace first conv: 3-channel RGB -> 1-channel mel spectrogram (randomly initialized)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("✅ EfficientNetB0Audio architecture defined (pretrained ImageNet backbone)")

✅ EfficientNetB0Audio architecture defined (pretrained ImageNet backbone)


In [9]:
# === CLEAN DATASET (No SpecAugment — proven to hurt performance) ===
# v6 used time/freq masking (30% chance each) which caused the same regression as Phase 1.
# v7 uses clean data with only random time crop during training.

class ClipDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, mel_root: str, cfg: dict, train: bool):
        self.df = frame.reset_index(drop=True)
        self.mel_root = Path(mel_root)
        self.cfg = cfg
        self.train = train
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]

        # Load mel — soundscape filenames are stored without .npy extension
        fname = str(r["filename"])
        if not fname.endswith(".npy"):
            mel_name = fname.replace("/", "_") + ".npy"
        else:
            mel_name = fname
        mel_path = self.mel_root / mel_name
        mel = np.load(mel_path).astype("float32")

        T = mel.shape[1]
        W = self.win_frames

        if T <= W:
            pad = np.zeros((mel.shape[0], W - T), dtype=np.float32)
            mel = np.concatenate([mel, pad], axis=1)
        else:
            # Random crop during training, center crop during validation
            start = np.random.randint(0, T - W) if self.train else max(0, (T - W) // 2)
            mel = mel[:, start:start + W]

        mel = np.clip(mel, 0.0, 1.0).astype(np.float32)
        x = torch.from_numpy(mel)[None, ...]
        y = torch.from_numpy(row_to_multihot(r["primary_label"], r.get("secondary_labels", "[]"))).float()
        return x.float(), y

print("\u2705 ClipDataset defined (no SpecAugment, random time crop only)")

✅ ClipDataset defined (no SpecAugment, random time crop only)


In [10]:
# === PREPARE TRAINING DATA ===
device = torch.device(CFG["device"])

mel_root = Path(MEL_OUT_DIR)
available_mels = set(f.stem for f in mel_root.glob("*.npy"))

print(f"Available mel files: {len(available_mels)}")

train_df = df.copy()
train_df["filename"] = train_df["filename"].apply(lambda x: x.replace("/", "_"))
train_df = train_df[train_df["filename"].isin(available_mels)].reset_index(drop=True)
print(f"Training samples from train_audio: {len(train_df)}")

# Merge soundscape pseudo-labeled data if available
if len(pseudo_df) > 0:
    train_df = pd.concat([train_df, pseudo_df], ignore_index=True)
    print(f"Added {len(pseudo_df)} soundscape segments for missing/underrepresented species")
    print(f"Total training samples: {len(train_df)}")
else:
    print("No soundscape segments added")

# Count species occurrences for class weighting
species_counts = {sp: 0 for sp in species}
for _, row in train_df.iterrows():
    primary = str(row["primary_label"])
    if primary in species_counts:
        species_counts[primary] += 1
    for sp in parse_secondary(row.get("secondary_labels", "[]")):
        if sp in species_counts:
            species_counts[sp] += 1

print(f"\n\u2705 Training setup complete:")
print(f"  Device: {device}")
print(f"  Classes: {n_classes}")
print(f"  Samples: {len(train_df)}")
print(f"  Species with data: {sum(1 for c in species_counts.values() if c > 0)}")

Available mel files: 35549
Training samples from train_audio: 35549
No soundscape segments added

✅ Training setup complete:
  Device: cuda
  Classes: 234
  Samples: 35549
  Species with data: 206


In [11]:
# === 5-FOLD CV TRAINING: RESNET18 ===
print("\n" + "=" * 70)
print("TRAINING: ResNet18Audio (replaces weak PerchAudio from v6)")
print("=" * 70)

all_scores_resnet = []
all_val_preds_resnet = []
all_val_targets = []

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG["seed"])

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    print(f"\n🔄 Fold {fold_idx + 1}/5")

    train_fold = train_df.iloc[train_idx]
    val_fold = train_df.iloc[val_idx]

    train_ds = ClipDataset(train_fold, MEL_OUT_DIR, CFG, train=True)
    val_ds   = ClipDataset(val_fold,   MEL_OUT_DIR, CFG, train=False)

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=CFG["num_workers"])
    val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"])

    model = ResNet18Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)

    # Class-balanced loss weights (capped at 10.0 to prevent extreme over-weighting of rare species)
    # Uncapped values can reach >4000x for rare species, causing AUC collapse (0.648 -> 0.472)
    pos_weight = torch.ones(n_classes).to(device)
    for i_sp, sp in enumerate(species):
        pos_weight[i_sp] = min(10.0, len(train_df) / (3.0 * max(1, species_counts[sp])))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = AdamW(model.parameters(), lr=CFG["lr"])
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-5)

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(CFG["epochs"]):
        # Training
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        scheduler.step()

        # Validation
        model.eval()
        val_loss = 0.0
        fold_preds, fold_targets = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()
                fold_preds.append(torch.sigmoid(logits).cpu().numpy())
                fold_targets.append(y.cpu().numpy())
        val_loss /= len(val_loader)

        # Compute validation AUC for monitoring
        fp = np.vstack(fold_preds)
        ft = np.vstack(fold_targets)
        auc_scores = []
        for j in range(n_classes):
            if ft[:, j].sum() > 0 and (1 - ft[:, j]).sum() > 0:
                auc_scores.append(roc_auc_score(ft[:, j], fp[:, j]))
        val_auc = np.mean(auc_scores) if auc_scores else 0.0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            best_fold_preds = fp
            best_fold_targets = ft
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or patience_counter == 0:
            print(f"  Epoch {epoch+1:3d}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_auc={val_auc:.4f}")

        if patience_counter >= CFG["patience"]:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)
    torch.save(model.state_dict(), f"/kaggle/working/resnet18_v7_fold_{fold_idx}.pt")
    all_scores_resnet.append(best_val_loss)
    all_val_preds_resnet.append(best_fold_preds)
    all_val_targets.append(best_fold_targets)
    print(f"  ✅ Best val_loss: {best_val_loss:.4f}")

print(f"\n✅ ResNet18 Training Complete: mean_loss={np.mean(all_scores_resnet):.4f} ± {np.std(all_scores_resnet):.4f}")


TRAINING: ResNet18Audio (replaces weak PerchAudio from v6)

🔄 Fold 1/5


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


  Epoch   1: train_loss=0.1707  val_loss=0.1442  val_auc=0.7929
  Epoch   2: train_loss=0.1394  val_loss=0.1337  val_auc=0.8440
  Epoch   3: train_loss=0.1269  val_loss=0.1264  val_auc=0.8627
  Epoch   4: train_loss=0.1177  val_loss=0.1191  val_auc=0.8863
  Epoch   5: train_loss=0.1100  val_loss=0.1113  val_auc=0.9000
  Epoch   6: train_loss=0.1029  val_loss=0.1106  val_auc=0.9025
  Epoch   8: train_loss=0.0896  val_loss=0.1027  val_auc=0.9127
  Epoch   9: train_loss=0.0821  val_loss=0.0984  val_auc=0.9200
  Epoch  10: train_loss=0.0745  val_loss=0.1038  val_auc=0.9151
  Epoch  15: train_loss=0.0368  val_loss=0.1331  val_auc=0.9149
  Early stopping at epoch 16
  ✅ Best val_loss: 0.0984

🔄 Fold 2/5
  Epoch   1: train_loss=0.1713  val_loss=0.1574  val_auc=0.7450
  Epoch   2: train_loss=0.1405  val_loss=0.1336  val_auc=0.8329
  Epoch   3: train_loss=0.1274  val_loss=0.1254  val_auc=0.8660
  Epoch   4: train_loss=0.1185  val_loss=0.1231  val_auc=0.8755
  Epoch   5: train_loss=0.1105  val_l

In [12]:
# === 5-FOLD CV TRAINING: EFFICIENTNET-B0 ===
print("\n" + "=" * 70)
print("TRAINING: EfficientNetB0Audio")
print("=" * 70)

all_scores_efficientnet = []
all_val_preds_efficientnet = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["primary_label"])):
    print(f"\n🔄 Fold {fold_idx + 1}/5")

    train_fold = train_df.iloc[train_idx]
    val_fold   = train_df.iloc[val_idx]

    train_ds = ClipDataset(train_fold, MEL_OUT_DIR, CFG, train=True)
    val_ds   = ClipDataset(val_fold,   MEL_OUT_DIR, CFG, train=False)

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=CFG["num_workers"])
    val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"])

    model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)

    # Class-balanced loss weights (capped at 10.0 to prevent extreme over-weighting of rare species)
    # Uncapped values can reach >4000x for rare species, causing AUC collapse (0.648 -> 0.472)
    pos_weight = torch.ones(n_classes).to(device)
    for i_sp, sp in enumerate(species):
        pos_weight[i_sp] = min(10.0, len(train_df) / (3.0 * max(1, species_counts[sp])))
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = AdamW(model.parameters(), lr=CFG["lr"])
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-5)

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(CFG["epochs"]):
        model.train()
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        scheduler.step()

        model.eval()
        val_loss = 0.0
        fold_preds, fold_targets = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()
                fold_preds.append(torch.sigmoid(logits).cpu().numpy())
                fold_targets.append(y.cpu().numpy())
        val_loss /= len(val_loader)

        fp = np.vstack(fold_preds)
        ft = np.vstack(fold_targets)
        auc_scores = []
        for j in range(n_classes):
            if ft[:, j].sum() > 0 and (1 - ft[:, j]).sum() > 0:
                auc_scores.append(roc_auc_score(ft[:, j], fp[:, j]))
        val_auc = np.mean(auc_scores) if auc_scores else 0.0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
            best_fold_preds = fp
            best_fold_targets = ft
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or patience_counter == 0:
            print(f"  Epoch {epoch+1:3d}: train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_auc={val_auc:.4f}")

        if patience_counter >= CFG["patience"]:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_model_state)
    torch.save(model.state_dict(), f"/kaggle/working/efficientnet_v7_fold_{fold_idx}.pt")
    all_scores_efficientnet.append(best_val_loss)
    all_val_preds_efficientnet.append(best_fold_preds)
    print(f"  ✅ Best val_loss: {best_val_loss:.4f}")

print(f"\n✅ EfficientNet Training Complete: mean_loss={np.mean(all_scores_efficientnet):.4f} ± {np.std(all_scores_efficientnet):.4f}")


TRAINING: EfficientNetB0Audio

🔄 Fold 1/5
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
100%|██████████| 20.5M/20.5M [00:00<00:00, 159MB/s]


  Epoch   1: train_loss=0.1586  val_loss=0.1206  val_auc=0.8652
  Epoch   2: train_loss=0.1153  val_loss=0.1020  val_auc=0.9113
  Epoch   3: train_loss=0.1007  val_loss=0.0944  val_auc=0.9215
  Epoch   4: train_loss=0.0911  val_loss=0.0914  val_auc=0.9272
  Epoch   5: train_loss=0.0828  val_loss=0.0886  val_auc=0.9332
  Epoch   6: train_loss=0.0760  val_loss=0.0883  val_auc=0.9346
  Epoch   7: train_loss=0.0691  val_loss=0.0855  val_auc=0.9421
  Epoch  10: train_loss=0.0489  val_loss=0.0901  val_auc=0.9419
  Early stopping at epoch 14
  ✅ Best val_loss: 0.0855

🔄 Fold 2/5
  Epoch   1: train_loss=0.1576  val_loss=0.1216  val_auc=0.8692
  Epoch   2: train_loss=0.1157  val_loss=0.1035  val_auc=0.9079
  Epoch   3: train_loss=0.1004  val_loss=0.0965  val_auc=0.9196
  Epoch   4: train_loss=0.0906  val_loss=0.0935  val_auc=0.9284
  Epoch   5: train_loss=0.0828  val_loss=0.0916  val_auc=0.9336
  Epoch   6: train_loss=0.0758  val_loss=0.0886  val_auc=0.9369
  Epoch   8: train_loss=0.0617  val_l

In [13]:
# === COMPUTE THRESHOLDS (Uniform 0.5 — per-species F1 proven not to generalize) ===
# v5/v6 used per-species F1-optimized thresholds -> caused 0.648->0.559 regression in Phase 1.
# v7 returns to the proven uniform 0.5 threshold approach.
print("\n" + "=" * 70)
print("THRESHOLD CONFIGURATION - Uniform 0.5 (proven approach)")
print("=" * 70)

# Compute ensemble AUC for reporting purposes only (not for threshold tuning)
all_resnet_preds = np.vstack(all_val_preds_resnet)
all_eff_preds    = np.vstack(all_val_preds_efficientnet)
all_targets      = np.vstack(all_val_targets)

ensemble_preds = (all_resnet_preds + all_eff_preds) / 2.0

auc_scores = []
for j in range(n_classes):
    y_true = all_targets[:, j]
    y_score = ensemble_preds[:, j]
    if y_true.sum() > 0 and (1 - y_true).sum() > 0:
        auc_scores.append(roc_auc_score(y_true, y_score))
mean_auc = np.mean(auc_scores) if auc_scores else 0.0

print(f"2-Model Ensemble Validation AUC: {mean_auc:.4f} (across {len(auc_scores)} species)")

# Uniform 0.5 thresholds for all species
optimal_thresholds = {sp: 0.5 for sp in species}

with open("/kaggle/working/optimal_thresholds_v7.json", "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

print(f"\u2705 Saved uniform thresholds (0.5) for {len(optimal_thresholds)} species")
print(f"   -> /kaggle/working/optimal_thresholds_v7.json")


THRESHOLD CONFIGURATION - Uniform 0.5 (proven approach)
2-Model Ensemble Validation AUC: 0.9346 (across 206 species)
✅ Saved uniform thresholds (0.5) for 234 species
   -> /kaggle/working/optimal_thresholds_v7.json


In [14]:
# === TRAINING SUMMARY ===
print("\n" + "=" * 70)
print("TRAINING SUMMARY - v7 (ResNet18 + EfficientNet-B0, Fixed Ensemble)")
print("=" * 70)

print(f"\n\U0001f4ca RESNET18 RESULTS:")
print(f"  Mean Val Loss: {np.mean(all_scores_resnet):.4f} \u00b1 {np.std(all_scores_resnet):.4f}")

print(f"\n\U0001f4ca EFFICIENTNET-B0 RESULTS:")
print(f"  Mean Val Loss: {np.mean(all_scores_efficientnet):.4f} \u00b1 {np.std(all_scores_efficientnet):.4f}")

print(f"\n\U0001f4ca 2-MODEL ENSEMBLE AUC: {mean_auc:.4f}")

print(f"\n\u2705 KEY FIXES vs v6:")
print(f"  \u2713 Replaced weak PerchAudio (3-layer CNN) with proven ResNet18")
print(f"  \u2713 Removed SpecAugment (time/freq masking proven to hurt)")
print(f"  \u2713 Fixed soundscape CSV filename (train_soundscapes_labels.csv)")
print(f"  \u2713 Targeted soundscape extraction for missing/underrepresented species")
print(f"  \u2713 Fixed EfficientNet model save (copy.deepcopy instead of shallow .copy())")
print(f"  \u2713 Added AUC metric tracking per epoch")
print(f"  \u2713 Cosine annealing LR scheduler")
print(f"  \u2713 Uniform 0.5 thresholds (not per-species F1)")

print(f"\n\U0001f4be SAVED ARTIFACTS:")
print(f"  - ResNet18:    resnet18_v7_fold_0-4.pt")
print(f"  - EfficientNet: efficientnet_v7_fold_0-4.pt")
print(f"  - Thresholds:  optimal_thresholds_v7.json")
print(f"  - Species:     species.json")

print(f"\n\u2705 Training Complete! Ready for v7 inference.")


TRAINING SUMMARY - v7 (ResNet18 + EfficientNet-B0, Fixed Ensemble)

📊 RESNET18 RESULTS:
  Mean Val Loss: 0.1005 ± 0.0012

📊 EFFICIENTNET-B0 RESULTS:
  Mean Val Loss: 0.0845 ± 0.0014

📊 2-MODEL ENSEMBLE AUC: 0.9346

✅ KEY FIXES vs v6:
  ✓ Replaced weak PerchAudio (3-layer CNN) with proven ResNet18
  ✓ Removed SpecAugment (time/freq masking proven to hurt)
  ✓ Fixed soundscape CSV filename (train_soundscapes_labels.csv)
  ✓ Targeted soundscape extraction for missing/underrepresented species
  ✓ Fixed EfficientNet model save (copy.deepcopy instead of shallow .copy())
  ✓ Added AUC metric tracking per epoch
  ✓ Cosine annealing LR scheduler
  ✓ Uniform 0.5 thresholds (not per-species F1)

💾 SAVED ARTIFACTS:
  - ResNet18:    resnet18_v7_fold_0-4.pt
  - EfficientNet: efficientnet_v7_fold_0-4.pt
  - Thresholds:  optimal_thresholds_v7.json
  - Species:     species.json

✅ Training Complete! Ready for v7 inference.
